**This code present ChatGPT's ability to annotate the test set for the original dataset (Feminism).**

In [ ]:
pip install openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 3.1 MB/s eta 0:00:00


In [ ]:
import openai
import csv
import pandas as pd


In [ ]:
openai.api_key = 'OPENTAI_API_KEY'

In [ ]:
df = pd.read_csv('Real_test_feminism_285.csv') #load the test set
df.head()

,ID,Target,Tweet,Stance
0,10390,Feminist Movement,Feminists can TOTALLY wear makeup and don't te...,AGAINST
1,10391,Feminist Movement,"I searched for posts with ""feminist"" tag and s...",AGAINST
2,10392,Feminist Movement,I saw a little girl wearing a mustache from th...,AGAINST
3,10393,Feminist Movement,Women are taught to put their values into thei...,AGAINST
4,10394,Feminist Movement,If u link anti feminism with misogyny or inequ...,AGAINST


In [ ]:
#take subset of tweets each time: 100, 100, 85
Tweet=df['Tweet'].tail(85)#iloc[100:200]

In [ ]:
#annotate the tweets using ChatGPT
prompt='''
"classify the stance of the tweet towards the feminist, your output should be only one of the stances ( FAVOR, AGAINST, or NONE).

Tweet: "the time for gender equality is NOW! #SemST".
FAVOR

Tweet: " RT @isocynic: I support whatever team buck supports #truefan #socceroos #bodypositive #SemST"
NONE

Tweet: " @KrewellaJahan feminism does not equal woman's rights. Feminism is about vaulting one gender over another, NOT equality. #SemST"
AGAINST
'''
Tweets=[]
Stance=[]
for data in Tweet:
  response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": prompt+ "Tweet: "+ data }
        ],
        max_tokens=5,
        n=1,
        temperature=0.2,
    )
  generated_label = response['choices'][0]['message']['content'].strip()
  Tweets.append(data)
  Stance.append(generated_label)





In [ ]:
output_df = pd.DataFrame({'Tweet': Tweets, 'Label': Stance})

#save it to CSV file
output_df.to_csv('generate_tweets.csv', index=False)

In [ ]:
df_gen = pd.read_csv('generate_tweets.csv')
df_gen.head()

,Tweet,Label
0,Feminists can TOTALLY wear makeup and don't te...,FAVOR
1,"I searched for posts with ""feminist"" tag and s...",AGAINST
2,I saw a little girl wearing a mustache from th...,NONE
3,Women are taught to put their values into thei...,FAVOR
4,If u link anti feminism with misogyny or inequ...,AGAINST


In [ ]:
#compare
generat_labe= df_gen["Label"] #chatGPT predictions
orignal_label=df["Stance"]   #original labels


In [ ]:
matchi=0
for syn_label, gold_label in zip(generat_labe, orignal_label):
  if syn_label==gold_label:
    matchi+=1

#calculate the percentage of correct matches
accuracy = matchi / 285 *100
print("Accuracy: {:.2f}%".format(accuracy))



Accuracy: 73.33%
